# OmniGeoFusion — Stage 1: SSL Pre-training

Self-supervised pre-training on Netherlands multimodal data.

**Prerequisites:** Run `00_prepare_data.ipynb` first  
**Runtime:** T4 GPU (~15GB VRAM)  
**Duration:** ~6-8 hours (50 epochs)  
**Checkpoints:** Auto-saved to Google Drive every epoch  
**Auto-resume:** Restarts from last checkpoint if Colab disconnects

### Objectives
1. Cross-modal contrastive (optical ↔ SAR ↔ LiDAR ↔ Thermal)
2. Temporal contrastive (T1 ↔ T2 weighted by day gap)

In [ ]:
# ── Cell 1: Mount Drive + Check GPU ──
from google.colab import drive
drive.mount('/gdrive')

import torch, os
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

CKPT_DIR = '/gdrive/MyDrive/omnigeofusion/checkpoints/ssl'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs('/gdrive/MyDrive/omnigeofusion/checkpoints/finetune', exist_ok=True)

# Check if resuming from previous session
latest = f'{CKPT_DIR}/ssl_latest.pt'
if os.path.exists(latest):
    ckpt = torch.load(latest, map_location='cpu')
    print(f'\n⚠️  Checkpoint found — will auto-resume from epoch {ckpt["epoch"]}')
    print(f'   Best loss so far: {ckpt.get("best_loss", "N/A")}')
else:
    print('\n✅ No checkpoint — starting fresh')

In [ ]:
# ── Cell 2: Setup Virtual Environment ──
# Same approach as P1 — venv isolates dependencies
# TerraTorch FIRST → then force-reinstall torch
import subprocess, os

VENV_PYTHON = '/content/venv/bin/python3'
VENV_PIP    = '/content/venv/bin/pip'

os.system('apt-get install -y libgeos-dev libgdal-dev -q')
os.environ['MPLBACKEND'] = 'agg'

# Create venv
os.system('python3 -m venv /content/venv')
os.system('curl -sS https://bootstrap.pypa.io/get-pip.py | /content/venv/bin/python3')

def install(packages):
    r = subprocess.run(
        [VENV_PIP, 'install', '-q'] + packages,
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(f'Error: {r.stderr[-500:]}')
        return False
    return True

print('Step 1: terratorch (first — owns its dependencies)...')
install(['terratorch'])

print('Step 2: torch (force reinstall after terratorch)...')
install(['--force-reinstall', 'torch==2.2.0', 'torchvision==0.17.0'])

print('Step 3: dependencies...')
install([
    'transformers==4.40.0', 'timm', 'einops',
    'mlflow==2.14.3', 'psycopg2-binary', 'boto3',
    'rasterio', 'scipy', 'pyyaml',
    'huggingface_hub==0.20.3', 'pandas', 'tqdm',
    'pyproj',
])

print('Step 4: pin critical versions last...')
install([
    'numpy==1.26.4',
    'huggingface-hub==0.20.3',
    'protobuf==3.20.3',
    'setuptools==69.5.1',
])

# System python also needs boto3 + rasterio for S3 ops
os.system('pip install -q boto3 rasterio')

r = subprocess.run([VENV_PYTHON, '-c', '''
import os; os.environ["MPLBACKEND"] = "agg"
import numpy as np, torch, pkg_resources
import google.protobuf
from terratorch.registry import BACKBONE_REGISTRY
print(f"numpy:      {np.__version__}")
print(f"torch:      {torch.__version__}")
print(f"protobuf:   {google.protobuf.__version__}")
print(f"terratorch: OK")
'''], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print('STDERR:', r.stderr[-200:])
print('✅ Cell 2 complete — venv ready')

In [ ]:
# ── Cell 3: Setup Repo ──
import subprocess, os

VENV_PYTHON = '/content/venv/bin/python3'
REPO_DIR    = '/content/omnigeofusion'
REPO_DRIVE  = '/gdrive/MyDrive/omnigeofusion/repo'
BUCKET      = 'omnigeofusion-data-288528696055'

if os.path.exists(f'{REPO_DIR}/src'):
    print('Repo already in /content')
elif os.path.exists(f'{REPO_DRIVE}/src'):
    print('Copying repo from Drive...')
    os.system(f'cp -r {REPO_DRIVE} {REPO_DIR}')
else:
    print('Downloading repo from S3...')
    os.system(f'''aws s3 cp \
        s3://{BUCKET}/repo/omnigeofusion.tar.gz \
        /tmp/omnigeofusion.tar.gz''')
    os.makedirs(REPO_DIR, exist_ok=True)
    os.system('tar -xzf /tmp/omnigeofusion.tar.gz -C /content/')

print(f'\nRepo contents: {os.listdir(REPO_DIR)[:6]}')
print('✅ Repo ready')

In [ ]:
# ── Cell 4: AWS Credentials + MLflow ──
# Credentials stored as Colab secrets (not hardcoded)
import subprocess, os
from google.colab import userdata

VENV_PYTHON = '/content/venv/bin/python3'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')
MLFLOW_URI  = 'http://54.83.1.56:5000'

# Set in environment for current session
os.environ['AWS_ACCESS_KEY_ID']     = AWS_KEY
os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
os.environ['MLFLOW_TRACKING_URI']   = MLFLOW_URI

# Test MLflow via venv
r = subprocess.run([VENV_PYTHON, '-c', f'''
import os, mlflow
os.environ["AWS_ACCESS_KEY_ID"]     = "{AWS_KEY}"
os.environ["AWS_SECRET_ACCESS_KEY"] = "{AWS_SECRET}"
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"
mlflow.set_tracking_uri("{MLFLOW_URI}")
try:
    mlflow.set_experiment("omnigeofusion-ssl")
    print(f"✅ MLflow connected: {MLFLOW_URI}")
except Exception as e:
    mlflow.set_tracking_uri("file:///content/mlruns")
    mlflow.set_experiment("omnigeofusion-ssl")
    print(f"⚠️  MLflow offline — using local: {{e}}")
'''], capture_output=True, text=True)
print(r.stdout)

# Test S3
import boto3
s3 = boto3.client('s3', region_name='us-east-1')
resp = s3.list_objects_v2(
    Bucket='omnigeofusion-data-288528696055',
    Prefix='netherlands/sentinel2/', MaxKeys=1
)
print(f'✅ S3 connected')

In [ ]:
# ── Cell 5: Verify Prithvi-EO-2.0-300M Loads ──
import subprocess, os
from google.colab import userdata

VENV_PYTHON = '/content/venv/bin/python3'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')

r = subprocess.run([VENV_PYTHON, '-c', f'''
import os, torch
os.environ["MPLBACKEND"]            = "agg"
os.environ["AWS_ACCESS_KEY_ID"]     = "{AWS_KEY}"
os.environ["AWS_SECRET_ACCESS_KEY"] = "{AWS_SECRET}"
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"

from terratorch.registry import BACKBONE_REGISTRY

print("Loading Prithvi-EO-2.0-300M via TerraTorch...")
encoder = BACKBONE_REGISTRY.build(
    "prithvi_eo_v2_300",
    pretrained=True,
    num_frames=1,
    in_chans=6,
)
params = sum(p.numel() for p in encoder.parameters())
print(f"✅ Prithvi loaded: {{params/1e6:.0f}}M parameters")

# Test forward pass on CPU
x = torch.zeros(1, 6, 224, 224)
with torch.no_grad():
    out = encoder(x)

if isinstance(out, (list, tuple)):
    feat = out[-1]
else:
    feat = out

if feat.dim() == 3:
    emb = feat[:, 1:, :].mean(dim=1)
else:
    emb = feat.mean(dim=[2,3]) if feat.dim()==4 else feat

print(f"✅ Embedding shape: {{emb.shape}}")

# Test on GPU
if torch.cuda.is_available():
    encoder = encoder.cuda()
    with torch.no_grad():
        out_gpu = encoder(x.cuda())
    feat_gpu = out_gpu[-1] if isinstance(out_gpu,(list,tuple)) else out_gpu
    emb_gpu  = feat_gpu[:,1:,:].mean(1) if feat_gpu.dim()==3 else feat_gpu.mean([2,3])
    print(f"✅ GPU forward: {{emb_gpu.shape}}")
    print(f"   GPU: {{torch.cuda.get_device_name(0)}}")
else:
    print("⚠️  No GPU — check runtime")

del encoder
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("✅ Prithvi ready for training")
'''], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print('STDERR:', r.stderr[-300:])

In [ ]:
# ── Cell 6: Verify Data from 00_prepare_data.ipynb ──
import json
from pathlib import Path

DATA_DIR = Path('/content/omnigeofusion_data')

if not (DATA_DIR / 'train_index.json').exists():
    raise FileNotFoundError(
        '❌ train_index.json not found!\n'
        'Please run 00_prepare_data.ipynb first.'
    )

train_idx = json.loads((DATA_DIR / 'train_index.json').read_text())
val_idx   = json.loads((DATA_DIR / 'val_index.json').read_text())

print(f'✅ Train: {len(train_idx)} samples')
print(f'✅ Val:   {len(val_idx)} samples')

for key, label in [
    ('s2_t1_path',   'Sentinel-2'),
    ('sar_path',     'Sentinel-1'),
    ('lidar_path',   'LiDAR'),
    ('thermal_path', 'Thermal'),
]:
    count = sum(1 for s in train_idx if key in s)
    pct   = count / len(train_idx) * 100
    print(f'   {label}: {count}/{len(train_idx)} ({pct:.0f}%)')

s = train_idx[0]
print(f'\nSample: {s["patch_id"]} | area: {s["area"]}')

In [ ]:
# ── Cell 7: Update Training Config for Colab ──
import subprocess, yaml, os

VENV_PYTHON = '/content/venv/bin/python3'
os.chdir('/content/omnigeofusion')

train_cfg = yaml.safe_load(open('configs/training_config.yaml'))

train_cfg['checkpoints'] = {
    'ssl_dir':      '/gdrive/MyDrive/omnigeofusion/checkpoints/ssl',
    'finetune_dir': '/gdrive/MyDrive/omnigeofusion/checkpoints/finetune',
    'gdrive_path':  '/gdrive/MyDrive/omnigeofusion/checkpoints',
}
train_cfg['ssl']['epochs']     = 50
train_cfg['ssl']['batch_size'] = 8
train_cfg['mlflow'] = {
    'tracking_uri':    'http://54.83.1.56:5000',
    'experiment_name': 'omnigeofusion-ssl',
}
train_cfg['data'] = {
    'bucket':      'omnigeofusion-data-288528696055',
    'prefix':      'netherlands',
    'data_dir':    '/content/omnigeofusion_data',
    'max_samples': 400,
}

with open('configs/training_config.yaml', 'w') as f:
    yaml.dump(train_cfg, f, default_flow_style=False)

print('✅ Training config updated:')
print(f'   Epochs:     {train_cfg["ssl"]["epochs"]}')
print(f'   Batch size: {train_cfg["ssl"]["batch_size"]}')
print(f'   Checkpoint: {train_cfg["checkpoints"]["ssl_dir"]}')

In [ ]:
# ── Cell 8: Run SSL Pre-training ──
# Runs in venv (has terratorch + correct dependencies)
# Auto-resumes from ssl_latest.pt if Colab disconnected
import subprocess, os
from google.colab import userdata

VENV_PYTHON = '/content/venv/bin/python3'
AWS_KEY     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET  = userdata.get('AWS_SECRET_ACCESS_KEY')

CKPT_DIR = '/gdrive/MyDrive/omnigeofusion/checkpoints/ssl'
latest   = f'{CKPT_DIR}/ssl_latest.pt'

# Build command
cmd = [
    VENV_PYTHON, '-m', 'src.training.ssl_pretrain',
    '--model-config',  'configs/model_config.yaml',
    '--train-config',  'configs/training_config.yaml',
    '--data-dir',      '/content/omnigeofusion_data',
    '--max-samples',   '400',
]
if os.path.exists(latest):
    print(f'⚠️  Auto-resuming from {latest}')
    cmd += ['--resume', latest]
else:
    print('Starting fresh SSL pre-training...')

# Set environment
env = os.environ.copy()
env['MPLBACKEND']            = 'agg'
env['AWS_ACCESS_KEY_ID']     = AWS_KEY
env['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET
env['AWS_DEFAULT_REGION']    = 'us-east-1'
env['MLFLOW_TRACKING_URI']   = 'http://54.83.1.56:5000'
env['PYTHONPATH']            = '/content/omnigeofusion'

os.chdir('/content/omnigeofusion')

# Run with live output
import sys
process = subprocess.Popen(
    cmd, env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print(f'\nProcess exited with code: {process.returncode}')

In [ ]:
# ── Cell 9: Verify Checkpoint + MLflow ──
import os, subprocess
from google.colab import userdata

VENV_PYTHON = '/content/venv/bin/python3'
CKPT_DIR    = '/gdrive/MyDrive/omnigeofusion/checkpoints/ssl'

print('=== Checkpoint Files ===')
for f in sorted(os.listdir(CKPT_DIR)):
    size = os.path.getsize(f'{CKPT_DIR}/{f}') / 1e6
    print(f'  {f}: {size:.1f}MB')

r = subprocess.run([VENV_PYTHON, '-c', f'''
import torch, mlflow, os

ckpt_best = "{CKPT_DIR}/ssl_best.pt"
if os.path.exists(ckpt_best):
    ckpt = torch.load(ckpt_best, map_location="cpu")
    print(f"\n✅ Best checkpoint:")
    print(f"   Epoch:     {{ckpt.get('epoch', 'N/A')}}")
    print(f"   Best loss: {{ckpt.get('best_loss', 'N/A')}}")
else:
    print("⚠️  No best checkpoint yet")

mlflow.set_tracking_uri("http://54.83.1.56:5000")
try:
    runs = mlflow.search_runs(experiment_names=["omnigeofusion-ssl"])
    print(f"\nMLflow runs: {{len(runs)}}")
    if len(runs) > 0:
        print(runs[["run_id","status"]].head())
except Exception as e:
    print(f"MLflow: {{e}}")
'''], capture_output=True, text=True)
print(r.stdout)

In [ ]:
# ── Cell 10: Upload Best Checkpoint to S3 ──
import boto3, os
from google.colab import userdata

AWS_KEY    = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET = userdata.get('AWS_SECRET_ACCESS_KEY')

s3 = boto3.client('s3',
    region_name='us-east-1',
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET
)
BUCKET = 'omnigeofusion-data-288528696055'
best   = '/gdrive/MyDrive/omnigeofusion/checkpoints/ssl/ssl_best.pt'

if os.path.exists(best):
    size = os.path.getsize(best) / 1e6
    print(f'Uploading ssl_best.pt ({size:.1f}MB)...')
    s3.upload_file(best, BUCKET, 'models/ssl/ssl_best.pt')
    print(f'✅ s3://{BUCKET}/models/ssl/ssl_best.pt')
    print('\n→ Ready for fine-tuning: 02_finetune.ipynb')
else:
    print('⚠️  No checkpoint to upload yet')